# 02 - Data Cleaning and Preparation

## Project: Enterprise IT RAG-Based AI Document Assistant

### Objective

The objective of this notebook is to clean, standardize,
and validate the enterprise IT document dataset.

### Key Activities

1. Load the original raw dataset
2. Create a separate working copy
3. Standardize column names and metadata
4. Handle missing values
5. Identify and manage duplicate records
6. Standardize categorical values
7. Evaluate document quality and relevance
8. Validate the cleaned dataset
9. Export the cleaned dataset

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
current_path = Path.cwd()

project_root = None

for path in [current_path, *current_path.parents]:
    if (path / "data" / "raw").exists():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError("Project directory not found.")

data_dir = project_root / "data" / "raw"

excel_files = list(data_dir.glob("*.xlsx"))

if len(excel_files) != 1:
    raise ValueError(f"Expected one Excel file, found {len(excel_files)}.")

file_path = excel_files[0]

raw_df = pd.read_excel(
    file_path,
    sheet_name="Document_Corpus"
)

print("Dataset loaded successfully!")
print("Dataset shape:", raw_df.shape)

Dataset loaded successfully!
Dataset shape: (84, 14)


In [3]:
clean_df = raw_df.copy(deep=True)

print("Working copy created successfully!")

print("Original dataset shape:", raw_df.shape)
print("Working dataset shape:", clean_df.shape)

Working copy created successfully!
Original dataset shape: (84, 14)
Working dataset shape: (84, 14)


In [4]:
clean_df.columns.tolist()

['document_id',
 'title',
 'category',
 'department',
 'owner',
 'effective_date',
 'review_date',
 'version',
 'status',
 'source_system',
 'access_level',
 'document_text',
 'quality_flags',
 'expected_action']

In [5]:
clean_df.dtypes

document_id            str
title                  str
category               str
department             str
owner                  str
effective_date         str
review_date            str
version            float64
status                 str
source_system          str
access_level           str
document_text          str
quality_flags          str
expected_action        str
dtype: object

In [6]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   document_id      84 non-null     str    
 1   title            84 non-null     str    
 2   category         84 non-null     str    
 3   department       84 non-null     str    
 4   owner            77 non-null     str    
 5   effective_date   83 non-null     str    
 6   review_date      77 non-null     str    
 7   version          82 non-null     float64
 8   status           84 non-null     str    
 9   source_system    84 non-null     str    
 10  access_level     84 non-null     str    
 11  document_text    84 non-null     str    
 12  quality_flags    84 non-null     str    
 13  expected_action  84 non-null     str    
dtypes: float64(1), str(13)
memory usage: 9.3 KB


In [7]:
clean_df.head()

,document_id,title,category,department,owner,effective_date,review_date,version,status,source_system,access_level,document_text,quality_flags,expected_action
0,DOC-0001,Privileged Access Standard,Access Management,Security,Security Owner,2024-01-24,2025-01-23,1.1,Approved,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to security teams and authorized service owners. It des...,none,retain
1,DOC-0002,Privileged Access Standard - Working Copy 2,Access Management,Security,Security Owner,2024-02-02,2025-02-01,2.2,Draft,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to security teams and authorized service owners. It des...,none,retain
2,DOC-0003,Privileged Access Standard - Working Copy 3,Access Management,Security,Security Owner,2024-02-11,2025-02-10,3.0,Superseded,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to security teams and authorized service owners. It des...,none,retain
3,DOC-0004,Joiner-Mover-Leaver Procedure,Access Management,IT Operations,ITOperations Owner,2024-02-20,2025-02-19,1.1,Approved,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to it operations teams and authorized service owners. I...,none,retain
4,DOC-0005,Joiner-Mover-Leaver Procedure - Working Copy 2,Access Mgmt,IT Operations,ITOperations Owner,2024-02-29,2025-02-28,2.2,Draft,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to it operations teams and authorized service owners. I...,inconsistent_category,retain


In [8]:
clean_df.isnull().sum()

document_id        0
title              0
category           0
department         0
owner              7
effective_date     1
review_date        7
version            2
status             0
source_system      0
access_level       0
document_text      0
quality_flags      0
expected_action    0
dtype: int64

In [9]:

column_profile = pd.DataFrame({
    "column_name": clean_df.columns,
    "data_type": clean_df.dtypes.astype(str).values,
    "missing_count": clean_df.isnull().sum().values,
    "missing_percentage": (
        clean_df.isnull().mean().values * 100
    ).round(2),
    "unique_values": clean_df.nunique(dropna=True).values,
    "duplicate_count": [
        clean_df[column].duplicated().sum()
        for column in clean_df.columns
    ],
    "sample_value": [
        clean_df[column].dropna().iloc[0]
        if clean_df[column].notna().any()
        else None
        for column in clean_df.columns
    ]
})

column_profile


,column_name,data_type,missing_count,missing_percentage,unique_values,duplicate_count,sample_value
0,document_id,str,0,0.00,84,0,DOC-0001
1,title,str,0,0.00,82,2,Privileged Access Standard
2,category,str,0,0.00,19,65,Access Management
3,department,str,0,0.00,14,70,Security
4,owner,str,7,8.33,19,64,Security Owner
5,effective_date,str,1,1.19,83,0,2024-01-24
6,review_date,str,7,8.33,77,6,2025-01-23
7,version,float64,2,2.38,8,75,1.1
8,status,str,0,0.00,8,76,Approved
9,source_system,str,0,0.00,11,73,Controlled Knowledge Repository


In [10]:

report_dir = project_root / "reports" / "data_quality_report"
report_dir.mkdir(parents=True, exist_ok=True)

column_profile.to_csv(
    report_dir / "column_profile_before_cleaning.csv",
    index=False
)

print("Column profile saved successfully!")
print("Location:", report_dir)


Column profile saved successfully!
Location: C:\Users\main\Desktop\Projects\RAG Based AI Assistant\reports\data_quality_report


In [11]:

missing_columns = clean_df.columns[clean_df.isnull().any()].tolist()

print("Columns containing missing values:")
print(missing_columns)

missing_records = clean_df.loc[
    clean_df[missing_columns].notna().sum(axis=1) < len(missing_columns)
].copy()

missing_records


Columns containing missing values:
['owner', 'effective_date', 'review_date', 'version']


,document_id,title,category,department,owner,effective_date,review_date,version,status,source_system,access_level,document_text,quality_flags,expected_action
7,DOC-0008,Service Account Governance - Working Copy 2,Access Management,Security,NaN,2024-03-27,2025-03-27,2.2,Draft,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to security teams and authorized service owners. It des...,missing_owner,retain
11,DOC-0012,Incident Priority and Escalation Matrix - Working Copy 3,Incident Management,IT Service Desk,ITServiceDesk Owner,2024-05-02,NaN,3.0,Superseded,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to it service desk teams and authorized service owners....,missing_review_date,retain
26,DOC-0027,Vendor Support Escalation Guide - Working Copy 3,Service Levels,Vendor Management,NaN,2024-09-14,2025-09-14,3.0,Superseded,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to vendor management teams and authorized service owner...,missing_owner,retain
34,DOC-0035,Change Freeze Calendar Guidance - Working Copy 2,Change Management,Application Support,ApplicationSupport Owner,2024-11-25,NaN,2.2,Draft,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to application support teams and authorized service own...,missing_review_date,retain
45,DOC-0046,Cloud Resource Provisioning Standard,Network and Cloud,Cloud Platform,NaN,2025-03-04,2026-03-04,1.1,Approved,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to cloud platform teams and authorized service owners. ...,missing_owner,retain
57,DOC-0058,Secure File Transfer Procedure,Data Protection,Security,Security Owner,2025-06-20,NaN,1.1,Approved,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to security teams and authorized service owners. It des...,missing_review_date,retain
64,DOC-0065,Managed Laptop Troubleshooting Guide - Working Copy 2,End User Support,IT Service Desk,NaN,2025-08-22,2026-08-22,2.2,Draft,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to it service desk teams and authorized service owners....,missing_owner,retain
72,DOC-X001,Privileged Access Standard,Access Mgmt,IT Ops,NaN,NaN,NaN,NaN,Active,SharePoint export,Internal,Privileged access can be granted by any team member without manager or security approval. This s...,inaccurate_content; inconsistent_category; missing_owner; missing_dates; missing_version; mislea...,quarantine
78,DOC-X007,Cloud Resource Provisioning Standard,Cloud Platform,Cloud Platform,Cloud Owner,2025-07-01,NaN,2.1,Approved,Cloud Wiki,Internal,"Cloud resources must use approved landing zones, identity controls, encryption, logging, tagging...",missing_review_date; partial_content,retain_with_metadata_warning
79,DOC-X008,Password reset quick note,End User Support,IT Service Desk,NaN,2025-08-01,NaN,NaN,Published,Ticket comment,Internal,"If a user cannot sign in, verify identity using the approved process and follow the identity pla...",missing_owner; missing_review_date; missing_version; weak_provenance,retain_with_warning


In [12]:

duplicate_titles = clean_df[
    clean_df["title"].duplicated(keep=False)
].sort_values("title")

print("Number of records with repeated titles:", len(duplicate_titles))

duplicate_titles[
    ["document_id", "title", "category", "department", "version", "status"]
]


Number of records with repeated titles: 4


,document_id,title,category,department,version,status
45,DOC-0046,Cloud Resource Provisioning Standard,Network and Cloud,Cloud Platform,1.1,Approved
78,DOC-X007,Cloud Resource Provisioning Standard,Cloud Platform,Cloud Platform,2.1,Approved
0,DOC-0001,Privileged Access Standard,Access Management,Security,1.1,Approved
72,DOC-X001,Privileged Access Standard,Access Mgmt,IT Ops,NaN,Active


In [13]:

categorical_columns = [
    "category",
    "department",
    "owner",
    "status",
    "source_system",
    "access_level"
]

for column in categorical_columns:
    if column in clean_df.columns:
        print(f"\n{'=' * 60}")
        print(f"Column: {column}")
        print(f"Unique values: {clean_df[column].nunique(dropna=True)}")
        print(clean_df[column].value_counts(dropna=False).to_string())



Column: category
Unique values: 19
category
Change Management           10
End User Support            10
Incident Management          9
Service Levels               9
Network and Cloud            9
Data Protection              9
Access Management            8
Backup and Recovery          8
Access Mgmt                  2
SLA                          1
BCP / DR                     1
Data Governance              1
Incident Mgmt                1
Facilities                   1
Communications               1
Network & Infrastructure     1
Cloud Platform               1
Vendor Material              1
Backup & Recovery            1

Column: department
Unique values: 14
department
Security               15
IT Service Desk        15
IT Operations          13
Business Continuity     7
Cloud Platform          7
Network Operations      7
Application Support     6
Compliance              4
Vendor Management       3
Infrastructure          3
IT Ops                  1
Facilities              1
HR   

In [14]:

category_profile = []

for column in categorical_columns:
    if column in clean_df.columns:
        counts = clean_df[column].value_counts(dropna=False)

        for value, count in counts.items():
            category_profile.append({
                "column_name": column,
                "value": str(value),
                "record_count": count
            })

category_profile_df = pd.DataFrame(category_profile)

category_profile_df.to_csv(
    report_dir / "categorical_value_profile.csv",
    index=False
)

print("Categorical profiling report saved successfully!")


Categorical profiling report saved successfully!


In [15]:

review_values = {
    "category": [
        "Access Mgmt",
        "Incident Mgmt",
        "SLA",
        "BCP / DR",
        "Network & Infrastructure",
        "Backup & Recovery"
    ],
    "department": ["IT Ops"],
    "status": ["approved"]
}

review_parts = []

for column, values in review_values.items():
    selected = clean_df[clean_df[column].isin(values)].copy()

    if not selected.empty:
        selected["field_under_review"] = column
        selected["value_under_review"] = selected[column]
        review_parts.append(selected)

if review_parts:
    metadata_review_df = pd.concat(review_parts, ignore_index=True)

    display(
        metadata_review_df[
            [
                "document_id",
                "title",
                "category",
                "department",
                "owner",
                "status",
                "source_system",
                "field_under_review",
                "value_under_review"
            ]
        ].sort_values(["field_under_review", "value_under_review"])
    )
else:
    print("No candidate values found.")


,document_id,title,category,department,owner,status,source_system,field_under_review,value_under_review
0,DOC-0005,Joiner-Mover-Leaver Procedure - Working Copy 2,Access Mgmt,IT Operations,ITOperations Owner,Draft,Controlled Knowledge Repository,category,Access Mgmt
3,DOC-X001,Privileged Access Standard,Access Mgmt,IT Ops,NaN,Active,SharePoint export,category,Access Mgmt
2,DOC-0039,Backup Operations Standard - Working Copy 3,BCP / DR,Infrastructure,Infrastructure Owner,Superseded,Controlled Knowledge Repository,category,BCP / DR
6,DOC-X010,Recovery Plan - Payments old copy,Backup & Recovery,Business Continuity,BCP Owner,Archived,Email Attachment,category,Backup & Recovery
4,DOC-X003,Incident Priority Matrix,Incident Mgmt,IT Service Desk,Service Desk Lead,Approved,Email Attachment,category,Incident Mgmt
5,DOC-X006,VPN Guide duplicate export,Network & Infrastructure,Network Operations,Network Owner,Approved,File Share,category,Network & Infrastructure
1,DOC-0022,Critical Application Support Agreement,SLA,Application Support,ApplicationSupport Owner,Approved,Controlled Knowledge Repository,category,SLA
7,DOC-X001,Privileged Access Standard,Access Mgmt,IT Ops,NaN,Active,SharePoint export,department,IT Ops


In [16]:

approved_review = clean_df[
    clean_df["status"].astype("string").str.strip().str.lower() == "approved"
]

display(
    approved_review[
        [
            "document_id",
            "title",
            "category",
            "department",
            "owner",
            "status",
            "source_system"
        ]
    ]
)


,document_id,title,category,department,owner,status,source_system
0,DOC-0001,Privileged Access Standard,Access Management,Security,Security Owner,Approved,Controlled Knowledge Repository
3,DOC-0004,Joiner-Mover-Leaver Procedure,Access Management,IT Operations,ITOperations Owner,Approved,Controlled Knowledge Repository
6,DOC-0007,Service Account Governance,Access Management,Security,Security Owner,Approved,Controlled Knowledge Repository
9,DOC-0010,Incident Priority and Escalation Matrix,Incident Management,IT Service Desk,ITServiceDesk Owner,Approved,Controlled Knowledge Repository
12,DOC-0013,Major Incident Communications SOP,Incident Management,IT Operations,ITOperations Owner,Approved,Controlled Knowledge Repository
15,DOC-0016,Endpoint Malware Response Runbook,Incident Management,Security,Security Owner,Approved,Controlled Knowledge Repository
18,DOC-0019,IT Service Desk SLA,Service Levels,IT Service Desk,ITServiceDesk Owner,Approved,Controlled Knowledge Repository
21,DOC-0022,Critical Application Support Agreement,SLA,Application Support,ApplicationSupport Owner,Approved,Controlled Knowledge Repository
24,DOC-0025,Vendor Support Escalation Guide,Service Levels,Vendor Management,VendorManagement Owner,Approved,Controlled Knowledge Repository
27,DOC-0028,Production Change Management Policy,Change Management,IT Operations,ITOperations Owner,Approved,Controlled Knowledge Repository


In [17]:

print("Status values with surrounding whitespace:")
print(
    clean_df["status"]
    .dropna()
    .map(repr)
    .unique()
)


Status values with surrounding whitespace:
<StringArray>
[  ''Approved'',      ''Draft'', ''Superseded'',  ''approved '',
     ''Active'',    ''Expired'',  ''Published'',   ''Archived'']
Length: 8, dtype: str


In [18]:

standardization_mapping = pd.DataFrame([
    {
        "column": "category",
        "original_value": "Access Mgmt",
        "standardized_value": "Access Management",
        "reason": "Abbreviated category label"
    },
    {
        "column": "category",
        "original_value": "Incident Mgmt",
        "standardized_value": "Incident Management",
        "reason": "Abbreviated category label"
    },
    {
        "column": "category",
        "original_value": "Backup & Recovery",
        "standardized_value": "Backup and Recovery",
        "reason": "Formatting variation"
    },
    {
        "column": "department",
        "original_value": "IT Ops",
        "standardized_value": "IT Operations",
        "reason": "Abbreviated department label"
    },
    {
        "column": "status",
        "original_value": "approved ",
        "standardized_value": "Approved",
        "reason": "Capitalization and trailing whitespace"
    }
])

display(standardization_mapping)


,column,original_value,standardized_value,reason
0,category,Access Mgmt,Access Management,Abbreviated category label
1,category,Incident Mgmt,Incident Management,Abbreviated category label
2,category,Backup & Recovery,Backup and Recovery,Formatting variation
3,department,IT Ops,IT Operations,Abbreviated department label
4,status,approved,Approved,Capitalization and trailing whitespace


In [19]:

standardization_mapping.to_csv(
    report_dir / "metadata_standardization_mapping.csv",
    index=False
)

print("Standardization mapping saved successfully!")


Standardization mapping saved successfully!


In [20]:

for _, row in standardization_mapping.iterrows():
    mask = clean_df[row["column"]] == row["original_value"]

    clean_df.loc[mask, row["column"]] = row["standardized_value"]

print("Approved metadata standardizations applied.")

print("\nUpdated category values:")
print(sorted(clean_df["category"].dropna().unique()))

print("\nUpdated department values:")
print(sorted(clean_df["department"].dropna().unique()))

print("\nUpdated status values:")
print(sorted(clean_df["status"].dropna().unique()))


Approved metadata standardizations applied.

Updated category values:
['Access Management', 'BCP / DR', 'Backup and Recovery', 'Change Management', 'Cloud Platform', 'Communications', 'Data Governance', 'Data Protection', 'End User Support', 'Facilities', 'Incident Management', 'Network & Infrastructure', 'Network and Cloud', 'SLA', 'Service Levels', 'Vendor Material']

Updated department values:
['Application Support', 'Business Continuity', 'Cloud Platform', 'Compliance', 'Facilities', 'HR', 'IT Operations', 'IT Service Desk', 'Infrastructure', 'Network Operations', 'Procurement', 'Security', 'Vendor Management']

Updated status values:
['Active', 'Approved', 'Archived', 'Draft', 'Expired', 'Published', 'Superseded']


In [21]:

missing_summary = pd.DataFrame({
    "column_name": clean_df.columns,
    "missing_count": clean_df.isnull().sum().values,
    "missing_percentage": (
        clean_df.isnull().mean().values * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_count", ascending=False)

display(missing_summary)


,column_name,missing_count,missing_percentage
4,owner,7,8.33
6,review_date,7,8.33
7,version,2,2.38
5,effective_date,1,1.19


In [22]:

metadata_columns = [
    "owner",
    "effective_date",
    "review_date",
    "version",
    "department",
    "category",
    "status"
]

available_metadata_columns = [
    column for column in metadata_columns
    if column in clean_df.columns
]

missing_metadata_records = clean_df.loc[
    clean_df[available_metadata_columns].isnull().any(axis=1),
    ["document_id", "title"] + available_metadata_columns
].copy()

print(
    "Number of documents with missing metadata:",
    len(missing_metadata_records)
)

display(missing_metadata_records)


Number of documents with missing metadata: 12


,document_id,title,owner,effective_date,review_date,version,department,category,status
7,DOC-0008,Service Account Governance - Working Copy 2,NaN,2024-03-27,2025-03-27,2.2,Security,Access Management,Draft
11,DOC-0012,Incident Priority and Escalation Matrix - Working Copy 3,ITServiceDesk Owner,2024-05-02,NaN,3.0,IT Service Desk,Incident Management,Superseded
26,DOC-0027,Vendor Support Escalation Guide - Working Copy 3,NaN,2024-09-14,2025-09-14,3.0,Vendor Management,Service Levels,Superseded
34,DOC-0035,Change Freeze Calendar Guidance - Working Copy 2,ApplicationSupport Owner,2024-11-25,NaN,2.2,Application Support,Change Management,Draft
45,DOC-0046,Cloud Resource Provisioning Standard,NaN,2025-03-04,2026-03-04,1.1,Cloud Platform,Network and Cloud,Approved
57,DOC-0058,Secure File Transfer Procedure,Security Owner,2025-06-20,NaN,1.1,Security,Data Protection,Approved
64,DOC-0065,Managed Laptop Troubleshooting Guide - Working Copy 2,NaN,2025-08-22,2026-08-22,2.2,IT Service Desk,End User Support,Draft
72,DOC-X001,Privileged Access Standard,NaN,NaN,NaN,NaN,IT Operations,Access Management,Active
78,DOC-X007,Cloud Resource Provisioning Standard,Cloud Owner,2025-07-01,NaN,2.1,Cloud Platform,Cloud Platform,Approved
79,DOC-X008,Password reset quick note,NaN,2025-08-01,NaN,NaN,IT Service Desk,End User Support,Published


In [23]:

duplicate_id_records = clean_df[
    clean_df["document_id"].duplicated(keep=False)
].sort_values("document_id")

print("Duplicate document ID records:", len(duplicate_id_records))

display(
    duplicate_id_records[
        ["document_id", "title", "version", "status", "source_system"]
    ]
)


Duplicate document ID records: 0


,document_id,title,version,status,source_system


In [24]:

duplicate_title_records = clean_df[
    clean_df["title"].duplicated(keep=False)
].sort_values("title")

print("Records with repeated titles:", len(duplicate_title_records))

display(
    duplicate_title_records[
        ["document_id", "title", "category", "version", "status", "source_system"]
    ]
)


Records with repeated titles: 4


,document_id,title,category,version,status,source_system
45,DOC-0046,Cloud Resource Provisioning Standard,Network and Cloud,1.1,Approved,Controlled Knowledge Repository
78,DOC-X007,Cloud Resource Provisioning Standard,Cloud Platform,2.1,Approved,Cloud Wiki
0,DOC-0001,Privileged Access Standard,Access Management,1.1,Approved,Controlled Knowledge Repository
72,DOC-X001,Privileged Access Standard,Access Management,NaN,Active,SharePoint export


In [25]:

exact_text_duplicates = clean_df[
    clean_df["document_text"].duplicated(keep=False)
].sort_values("document_text")

print("Records with identical document text:", len(exact_text_duplicates))

display(
    exact_text_duplicates[
        ["document_id", "title", "version", "status", "source_system"]
    ]
)


Records with identical document text: 72


,document_id,title,version,status,source_system
35,DOC-0036,Change Freeze Calendar Guidance - Working Copy 3,3.0,Superseded,Controlled Knowledge Repository
34,DOC-0035,Change Freeze Calendar Guidance - Working Copy 2,2.2,Draft,Controlled Knowledge Repository
33,DOC-0034,Change Freeze Calendar Guidance,1.1,Approved,Controlled Knowledge Repository
23,DOC-0024,Critical Application Support Agreement - Working Copy 3,3.0,Superseded,Controlled Knowledge Repository
22,DOC-0023,Critical Application Support Agreement - Working Copy 2,2.2,Draft,Controlled Knowledge Repository
...,...,...,...,...,...
8,DOC-0009,Service Account Governance - Working Copy 3,3.0,Superseded,Controlled Knowledge Repository
6,DOC-0007,Service Account Governance,1.1,Approved,Controlled Knowledge Repository
24,DOC-0025,Vendor Support Escalation Guide,1.1,Approved,Controlled Knowledge Repository
25,DOC-0026,Vendor Support Escalation Guide - Working Copy 2,2.2,Draft,Controlled Knowledge Repository


In [26]:

duplicate_id_records.to_csv(
    report_dir / "duplicate_document_ids.csv",
    index=False
)

duplicate_title_records.to_csv(
    report_dir / "duplicate_document_titles.csv",
    index=False
)

exact_text_duplicates.to_csv(
    report_dir / "exact_duplicate_document_text.csv",
    index=False
)

print("Duplicate analysis reports saved successfully!")


Duplicate analysis reports saved successfully!


In [27]:

text_column = clean_df["document_text"]

print("Total records:", len(clean_df))
print("Missing document texts:", text_column.isna().sum())
print("Empty document texts:", text_column.fillna("").str.strip().eq("").sum())
print("Unique non-empty document texts:", text_column.dropna().nunique())
print("Records flagged as duplicate text:", text_column.duplicated(keep=False).sum())


Total records: 84
Missing document texts: 0
Empty document texts: 0
Unique non-empty document texts: 36
Records flagged as duplicate text: 72


In [28]:

text_groups = (
    clean_df.groupby("document_text", dropna=False)
    .agg(
        record_count=("document_id", "count"),
        document_ids=("document_id", lambda values: ", ".join(values)),
        titles=("title", lambda values: " | ".join(values))
    )
    .reset_index()
)

repeated_text_groups = (
    text_groups[text_groups["record_count"] > 1]
    .sort_values("record_count", ascending=False)
)

print("Number of repeated text groups:", len(repeated_text_groups))

display(
    repeated_text_groups[
        ["record_count", "document_ids", "titles"]
    ].head(20)
)


Number of repeated text groups: 24


,record_count,document_ids,titles
10,3,"DOC-0034, DOC-0035, DOC-0036",Change Freeze Calendar Guidance | Change Freeze Calendar Guidance - Working Copy 2 | Change Free...
11,3,"DOC-0022, DOC-0023, DOC-0024",Critical Application Support Agreement | Critical Application Support Agreement - Working Copy 2...
12,3,"DOC-0040, DOC-0041, DOC-0042",Disaster Recovery Plan - Payments | Disaster Recovery Plan - Payments - Working Copy 2 | Disaste...
13,3,"DOC-0043, DOC-0044, DOC-0045",Recovery Exercise Report | Recovery Exercise Report - Working Copy 2 | Recovery Exercise Report ...
14,3,"DOC-0052, DOC-0053, DOC-0054",Cloud Incident Triage Guide | Cloud Incident Triage Guide - Working Copy 2 | Cloud Incident Tria...
15,3,"DOC-0046, DOC-0047, DOC-0048",Cloud Resource Provisioning Standard | Cloud Resource Provisioning Standard - Working Copy 2 | C...
16,3,"DOC-0061, DOC-0062, DOC-0063",Data Retention Schedule Guidance | Data Retention Schedule Guidance - Working Copy 2 | Data Rete...
17,3,"DOC-0037, DOC-0038, DOC-0039",Backup Operations Standard | Backup Operations Standard - Working Copy 2 | Backup Operations Sta...
18,3,"DOC-0031, DOC-0032, DOC-0033",Emergency Change Procedure | Emergency Change Procedure - Working Copy 2 | Emergency Change Proc...
19,3,"DOC-0004, DOC-0005, DOC-0006",Joiner-Mover-Leaver Procedure | Joiner-Mover-Leaver Procedure - Working Copy 2 | Joiner-Mover-Le...


In [29]:

for index, row in clean_df.head(5).iterrows():
    print("=" * 80)
    print("Document ID:", row["document_id"])
    print("Title:", row["title"])
    print("Text length:", len(str(row["document_text"])))
    print("Text preview:")
    print(str(row["document_text"])[:500])


Document ID: DOC-0001
Title: Privileged Access Standard
Text length: 1025
Text preview:
Purpose and scope: This document applies to security teams and authorized service owners. It describes the operational control and minimum evidence expected for privileged access standard.

Procedure / control: Privileged access requires a named business sponsor, manager approval, and security approval. Access must be time-bound, logged, and reviewed quarterly. Emergency access is permitted only through the break-glass process and must be reviewed by the next business day.

Operational evidence:
Document ID: DOC-0002
Title: Privileged Access Standard - Working Copy 2
Text length: 1025
Text preview:
Purpose and scope: This document applies to security teams and authorized service owners. It describes the operational control and minimum evidence expected for privileged access standard.

Procedure / control: Privileged access requires a named business sponsor, manager approval, and security approval. A

In [30]:

repeated_text_groups.drop(columns=["document_text"]).to_csv(
    report_dir / "repeated_document_text_groups.csv",
    index=False
)

print("Repeated-text group report saved successfully!")


Repeated-text group report saved successfully!


In [31]:

import hashlib

clean_df["content_hash"] = clean_df["document_text"].apply(
    lambda text: hashlib.sha256(
        str(text).encode("utf-8")
    ).hexdigest()
)

print("Content fingerprints generated successfully!")
print("Unique fingerprints:", clean_df["content_hash"].nunique())
print("Total records:", len(clean_df))


Content fingerprints generated successfully!
Unique fingerprints: 36
Total records: 84


In [32]:

hash_to_group = {
    content_hash: f"DUP-{number:03d}"
    for number, content_hash in enumerate(
        clean_df["content_hash"].drop_duplicates(),
        start=1
    )
}

clean_df["duplicate_group_id"] = clean_df["content_hash"].map(
    hash_to_group
)

group_sizes = clean_df.groupby("duplicate_group_id").size()

clean_df["duplicate_record_count"] = (
    clean_df["duplicate_group_id"].map(group_sizes)
)

clean_df["duplicate_type"] = np.where(
    clean_df["duplicate_record_count"] > 1,
    "Exact content match",
    "Unique content"
)

display(
    clean_df[
        [
            "document_id",
            "title",
            "version",
            "status",
            "duplicate_group_id",
            "duplicate_record_count",
            "duplicate_type"
        ]
    ].sort_values(["duplicate_group_id", "document_id"])
)


,document_id,title,version,status,duplicate_group_id,duplicate_record_count,duplicate_type
0,DOC-0001,Privileged Access Standard,1.1,Approved,DUP-001,3,Exact content match
1,DOC-0002,Privileged Access Standard - Working Copy 2,2.2,Draft,DUP-001,3,Exact content match
2,DOC-0003,Privileged Access Standard - Working Copy 3,3.0,Superseded,DUP-001,3,Exact content match
3,DOC-0004,Joiner-Mover-Leaver Procedure,1.1,Approved,DUP-002,3,Exact content match
4,DOC-0005,Joiner-Mover-Leaver Procedure - Working Copy 2,2.2,Draft,DUP-002,3,Exact content match
...,...,...,...,...,...,...,...
79,DOC-X008,Password reset quick note,NaN,Published,DUP-032,1,Unique content
80,DOC-X009,Vendor Product Brochure,1.0,Published,DUP-033,1,Unique content
81,DOC-X010,Recovery Plan - Payments old copy,1.2,Archived,DUP-034,1,Unique content
82,DOC-X011,Data Retention - contradictory note,0.9,Draft,DUP-035,1,Unique content


In [33]:

duplicate_audit = clean_df[
    [
        "document_id",
        "title",
        "category",
        "department",
        "owner",
        "version",
        "status",
        "source_system",
        "content_hash",
        "duplicate_group_id",
        "duplicate_record_count",
        "duplicate_type"
    ]
].copy()

duplicate_audit.to_csv(
    report_dir / "document_duplicate_audit.csv",
    index=False
)

print("Document duplicate audit saved successfully!")


Document duplicate audit saved successfully!


In [34]:

duplicate_group_summary = (
    clean_df.groupby("duplicate_group_id")
    .agg(
        record_count=("document_id", "count"),
        document_ids=("document_id", lambda values: ", ".join(values)),
        titles=("title", lambda values: " | ".join(values)),
        statuses=("status", lambda values: ", ".join(values.fillna("Missing")))
    )
    .reset_index()
)

duplicate_group_summary = duplicate_group_summary[
    duplicate_group_summary["record_count"] > 1
].sort_values("record_count", ascending=False)

display(duplicate_group_summary)

duplicate_group_summary.to_csv(
    report_dir / "duplicate_group_summary.csv",
    index=False
)


,duplicate_group_id,record_count,document_ids,titles,statuses
0,DUP-001,3,"DOC-0001, DOC-0002, DOC-0003",Privileged Access Standard | Privileged Access Standard - Working Copy 2 | Privileged Access Sta...,"Approved, Draft, Superseded"
1,DUP-002,3,"DOC-0004, DOC-0005, DOC-0006",Joiner-Mover-Leaver Procedure | Joiner-Mover-Leaver Procedure - Working Copy 2 | Joiner-Mover-Le...,"Approved, Draft, Superseded"
2,DUP-003,3,"DOC-0007, DOC-0008, DOC-0009",Service Account Governance | Service Account Governance - Working Copy 2 | Service Account Gover...,"Approved, Draft, Superseded"
3,DUP-004,3,"DOC-0010, DOC-0011, DOC-0012",Incident Priority and Escalation Matrix | Incident Priority and Escalation Matrix - Working Copy...,"Approved, Draft, Superseded"
4,DUP-005,3,"DOC-0013, DOC-0014, DOC-0015",Major Incident Communications SOP | Major Incident Communications SOP - Working Copy 2 | Major I...,"Approved, Draft, Superseded"
5,DUP-006,3,"DOC-0016, DOC-0017, DOC-0018",Endpoint Malware Response Runbook | Endpoint Malware Response Runbook - Working Copy 2 | Endpoin...,"Approved, Draft, Superseded"
6,DUP-007,3,"DOC-0019, DOC-0020, DOC-0021",IT Service Desk SLA | IT Service Desk SLA - Working Copy 2 | IT Service Desk SLA - Working Copy 3,"Approved, Draft, Superseded"
7,DUP-008,3,"DOC-0022, DOC-0023, DOC-0024",Critical Application Support Agreement | Critical Application Support Agreement - Working Copy 2...,"Approved, Draft, Superseded"
8,DUP-009,3,"DOC-0025, DOC-0026, DOC-0027",Vendor Support Escalation Guide | Vendor Support Escalation Guide - Working Copy 2 | Vendor Supp...,"Approved, Draft, Superseded"
9,DUP-010,3,"DOC-0028, DOC-0029, DOC-0030",Production Change Management Policy | Production Change Management Policy - Working Copy 2 | Pro...,"Approved, Draft, Superseded"


In [35]:

def classify_authority(row):
    status = str(row["status"]).strip().lower()
    source = str(row["source_system"]).strip().lower()

    if status == "approved" and source == "controlled knowledge repository":
        return "Approved Candidate"

    if status == "approved":
        return "External Approved"

    if status in ["draft", "superseded", "expired", "archived"]:
        return "Non-Approved"

    return "Other"

clean_df["authority_class"] = clean_df.apply(
    classify_authority,
    axis=1
)

print(clean_df["authority_class"].value_counts(dropna=False))


authority_class
Non-Approved          53
Approved Candidate    24
External Approved      4
Other                  3
Name: count, dtype: int64


In [36]:

authority_review = clean_df[
    [
        "document_id",
        "title",
        "version",
        "status",
        "source_system",
        "owner",
        "effective_date",
        "review_date",
        "duplicate_group_id",
        "authority_class"
    ]
].sort_values(["duplicate_group_id", "authority_class"])

display(authority_review.head(30))


,document_id,title,version,status,source_system,owner,effective_date,review_date,duplicate_group_id,authority_class
0,DOC-0001,Privileged Access Standard,1.1,Approved,Controlled Knowledge Repository,Security Owner,2024-01-24,2025-01-23,DUP-001,Approved Candidate
1,DOC-0002,Privileged Access Standard - Working Copy 2,2.2,Draft,Controlled Knowledge Repository,Security Owner,2024-02-02,2025-02-01,DUP-001,Non-Approved
2,DOC-0003,Privileged Access Standard - Working Copy 3,3.0,Superseded,Controlled Knowledge Repository,Security Owner,2024-02-11,2025-02-10,DUP-001,Non-Approved
3,DOC-0004,Joiner-Mover-Leaver Procedure,1.1,Approved,Controlled Knowledge Repository,ITOperations Owner,2024-02-20,2025-02-19,DUP-002,Approved Candidate
4,DOC-0005,Joiner-Mover-Leaver Procedure - Working Copy 2,2.2,Draft,Controlled Knowledge Repository,ITOperations Owner,2024-02-29,2025-02-28,DUP-002,Non-Approved
5,DOC-0006,Joiner-Mover-Leaver Procedure - Working Copy 3,3.0,Superseded,Controlled Knowledge Repository,ITOperations Owner,2024-03-09,2025-03-09,DUP-002,Non-Approved
6,DOC-0007,Service Account Governance,1.1,Approved,Controlled Knowledge Repository,Security Owner,2024-03-18,2025-03-18,DUP-003,Approved Candidate
7,DOC-0008,Service Account Governance - Working Copy 2,2.2,Draft,Controlled Knowledge Repository,NaN,2024-03-27,2025-03-27,DUP-003,Non-Approved
8,DOC-0009,Service Account Governance - Working Copy 3,3.0,Superseded,Controlled Knowledge Repository,Security Owner,2024-04-05,2025-04-05,DUP-003,Non-Approved
9,DOC-0010,Incident Priority and Escalation Matrix,1.1,Approved,Controlled Knowledge Repository,ITServiceDesk Owner,2024-04-14,2025-04-14,DUP-004,Approved Candidate


In [37]:

authority_review.to_csv(
    report_dir / "document_authority_assessment.csv",
    index=False
)

print("Document authority assessment saved successfully!")


Document authority assessment saved successfully!


In [38]:

authority_exceptions = clean_df[
    clean_df["authority_class"].isin([
        "External Approved",
        "Other"
    ])
].copy()

exception_columns = [
    "document_id",
    "title",
    "category",
    "department",
    "owner",
    "version",
    "status",
    "source_system",
    "effective_date",
    "review_date",
    "access_level",
    "duplicate_group_id",
    "authority_class"
]

display(
    authority_exceptions[exception_columns].sort_values(
        ["authority_class", "document_id"]
    )
)


,document_id,title,category,department,owner,version,status,source_system,effective_date,review_date,access_level,duplicate_group_id,authority_class
74,DOC-X003,Incident Priority Matrix,Incident Management,IT Service Desk,Service Desk Lead,2.0,Approved,Email Attachment,2025-01-01,2026-01-01,Internal,DUP-027,External Approved
77,DOC-X006,VPN Guide duplicate export,Network & Infrastructure,Network Operations,Network Owner,1.0,Approved,File Share,2025-06-01,2026-06-01,Internal,DUP-030,External Approved
78,DOC-X007,Cloud Resource Provisioning Standard,Cloud Platform,Cloud Platform,Cloud Owner,2.1,Approved,Cloud Wiki,2025-07-01,NaN,Internal,DUP-031,External Approved
83,DOC-X012,Production change approval FAQ,Change Management,IT Operations,NaN,1.0,Approved,Knowledge Base,2025-09-01,2026-09-01,Internal,DUP-036,External Approved
72,DOC-X001,Privileged Access Standard,Access Management,IT Operations,NaN,NaN,Active,SharePoint export,NaN,NaN,Internal,DUP-025,Other
79,DOC-X008,Password reset quick note,End User Support,IT Service Desk,NaN,NaN,Published,Ticket comment,2025-08-01,NaN,Internal,DUP-032,Other
80,DOC-X009,Vendor Product Brochure,Vendor Material,Procurement,Vendor,1.0,Published,Vendor Portal,2025-03-01,NaN,Public,DUP-033,Other


In [39]:

for _, row in authority_exceptions.iterrows():
    print("=" * 90)
    print("Document ID:", row["document_id"])
    print("Title:", row["title"])
    print("Status:", row["status"])
    print("Source:", row["source_system"])
    print("Authority:", row["authority_class"])
    print("Document text:")
    print(row["document_text"])
    print()


Document ID: DOC-X001
Title: Privileged Access Standard
Status: Active
Source: SharePoint export
Authority: Other
Document text:
Privileged access can be granted by any team member without manager or security approval. This statement is intentionally inaccurate and conflicts with the approved standard. It is an old copied note and must not be used as policy.

Document ID: DOC-X003
Title: Incident Priority Matrix
Status: Approved
Source: Email Attachment
Authority: External Approved
Document text:
P1 incidents should be acknowledged within 4 hours. This value is intentionally inaccurate and is not an approved service target. Confirm the controlled SLA before answering response-time questions.

Document ID: DOC-X006
Title: VPN Guide duplicate export
Status: Approved
Source: File Share
Authority: External Approved
Document text:
Confirm the user account is active, device compliance is healthy, time synchronization is correct, and the approved VPN client is installed. Record the exact erro

In [40]:

authority_exceptions[exception_columns + ["document_text"]].to_csv(
    report_dir / "authority_exception_review.csv",
    index=False
)

print("Authority exception review saved successfully!")


Authority exception review saved successfully!


In [45]:
def classify_retrieval_eligibility(row):
    document_id = row["document_id"]

    if document_id in ["DOC-X001", "DOC-X003"]:
        return "Excluded - Unreliable"

    if document_id == "DOC-X009":
        return "Excluded - Irrelevant"

    if document_id in ["DOC-X006", "DOC-X007", "DOC-X012"]:
        return "External Validation Required"

    if document_id == "DOC-X008":
        return "External Validation Required"

    if row["authority_class"] == "Approved Candidate":
        return "Eligible Candidate"

    return "Manual Review Required"


clean_df["retrieval_eligibility"] = clean_df.apply(
    classify_retrieval_eligibility,
    axis=1
)

In [42]:
eligibility_columns = [
    "document_id",
    "title",
    "status",
    "source_system",
    "authority_class",
    "retrieval_eligibility"
]

display(
    clean_df[
        clean_df["document_id"].isin([
            "DOC-X001",
            "DOC-X003",
            "DOC-X006",
            "DOC-X007",
            "DOC-X008",
            "DOC-X009",
            "DOC-X012"
        ])
    ][eligibility_columns].sort_values("document_id")
)

,document_id,title,status,source_system,authority_class,retrieval_eligibility
72,DOC-X001,Privileged Access Standard,Active,SharePoint export,Other,Excluded - Unreliable
74,DOC-X003,Incident Priority Matrix,Approved,Email Attachment,External Approved,Excluded - Unreliable
77,DOC-X006,VPN Guide duplicate export,Approved,File Share,External Approved,External Validation Required
78,DOC-X007,Cloud Resource Provisioning Standard,Approved,Cloud Wiki,External Approved,External Validation Required
79,DOC-X008,Password reset quick note,Published,Ticket comment,Other,External Validation Required
80,DOC-X009,Vendor Product Brochure,Published,Vendor Portal,Other,Excluded - Irrelevant
83,DOC-X012,Production change approval FAQ,Approved,Knowledge Base,External Approved,External Validation Required


In [43]:
eligibility_summary = (
    clean_df["retrieval_eligibility"]
    .value_counts(dropna=False)
    .rename_axis("retrieval_eligibility")
    .reset_index(name="document_count")
)

display(eligibility_summary)

,retrieval_eligibility,document_count
0,Manual Review Required,53
1,Eligible Candidate,24
2,External Validation Required,4
3,Excluded - Unreliable,2
4,Excluded - Irrelevant,1


In [44]:
eligibility_audit = clean_df[
    [
        "document_id",
        "title",
        "category",
        "department",
        "owner",
        "version",
        "status",
        "source_system",
        "effective_date",
        "review_date",
        "access_level",
        "authority_class",
        "retrieval_eligibility",
        "content_hash",
        "duplicate_group_id",
        "document_text"
    ]
].copy()

eligibility_audit.to_csv(
    report_dir / "retrieval_eligibility_audit.csv",
    index=False
)

print("Retrieval eligibility audit saved.")

Retrieval eligibility audit saved.


In [46]:
manual_review_df = clean_df[
    clean_df["retrieval_eligibility"] == "Manual Review Required"
].copy()

manual_review_columns = [
    "document_id",
    "title",
    "category",
    "department",
    "owner",
    "version",
    "status",
    "source_system",
    "effective_date",
    "review_date",
    "access_level",
    "authority_class",
    "duplicate_group_id"
]

display(
    manual_review_df[manual_review_columns].sort_values("document_id")
)

,document_id,title,category,department,owner,version,status,source_system,effective_date,review_date,access_level,authority_class,duplicate_group_id
1,DOC-0002,Privileged Access Standard - Working Copy 2,Access Management,Security,Security Owner,2.2,Draft,Controlled Knowledge Repository,2024-02-02,2025-02-01,Internal,Non-Approved,DUP-001
2,DOC-0003,Privileged Access Standard - Working Copy 3,Access Management,Security,Security Owner,3.0,Superseded,Controlled Knowledge Repository,2024-02-11,2025-02-10,Internal,Non-Approved,DUP-001
4,DOC-0005,Joiner-Mover-Leaver Procedure - Working Copy 2,Access Management,IT Operations,ITOperations Owner,2.2,Draft,Controlled Knowledge Repository,2024-02-29,2025-02-28,Internal,Non-Approved,DUP-002
5,DOC-0006,Joiner-Mover-Leaver Procedure - Working Copy 3,Access Management,IT Operations,ITOperations Owner,3.0,Superseded,Controlled Knowledge Repository,2024-03-09,2025-03-09,Internal,Non-Approved,DUP-002
7,DOC-0008,Service Account Governance - Working Copy 2,Access Management,Security,NaN,2.2,Draft,Controlled Knowledge Repository,2024-03-27,2025-03-27,Internal,Non-Approved,DUP-003
8,DOC-0009,Service Account Governance - Working Copy 3,Access Management,Security,Security Owner,3.0,Superseded,Controlled Knowledge Repository,2024-04-05,2025-04-05,Internal,Non-Approved,DUP-003
10,DOC-0011,Incident Priority and Escalation Matrix - Working Copy 2,Incident Management,IT Service Desk,ITServiceDesk Owner,2.2,Draft,Controlled Knowledge Repository,2024-04-23,2025-04-23,Internal,Non-Approved,DUP-004
11,DOC-0012,Incident Priority and Escalation Matrix - Working Copy 3,Incident Management,IT Service Desk,ITServiceDesk Owner,3.0,Superseded,Controlled Knowledge Repository,2024-05-02,NaN,Internal,Non-Approved,DUP-004
13,DOC-0014,Major Incident Communications SOP - Working Copy 2,Incident Management,IT Operations,ITOperations Owner,2.2,Draft,Controlled Knowledge Repository,2024-05-20,2025-05-20,Internal,Non-Approved,DUP-005
14,DOC-0015,Major Incident Communications SOP - Working Copy 3,Incident Management,IT Operations,ITOperations Owner,3.0,Superseded,Controlled Knowledge Repository,2024-05-29,2025-05-29,Internal,Non-Approved,DUP-005


In [47]:
manual_status_summary = (
    manual_review_df["status"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("status")
    .reset_index(name="document_count")
)

manual_source_summary = (
    manual_review_df["source_system"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("source_system")
    .reset_index(name="document_count")
)

display(manual_status_summary)
display(manual_source_summary)

,status,document_count
0,Draft,25
1,Superseded,24
2,Expired,3
3,Archived,1


,source_system,document_count
0,Controlled Knowledge Repository,48
1,Intranet,2
2,Legacy Wiki,1
3,Email Attachment,1
4,Personal Drive,1


In [48]:
manual_review_df.to_csv(
    report_dir / "manual_review_queue.csv",
    index=False
)

print("Manual review queue saved.")
print("Records requiring manual review:", len(manual_review_df))

Manual review queue saved.
Records requiring manual review: 53


In [49]:
def assign_review_disposition(row):
    document_id = row["document_id"]
    status = str(row["status"]).strip().lower()

    if document_id == "DOC-X011":
        return "Quarantine - Restricted/Conflict"

    if document_id in ["DOC-X004", "DOC-X005"]:
        return "Exclude - Irrelevant"

    if status == "draft":
        return "Exclude - Draft"

    if status == "superseded":
        return "Exclude - Superseded"

    if status == "expired":
        return "Exclude - Expired"

    if status == "archived":
        return "Exclude - Archived"

    return "Manual Investigation Required"


clean_df["review_disposition"] = clean_df.apply(
    assign_review_disposition,
    axis=1
)

In [50]:
reviewed_records = clean_df[
    clean_df["retrieval_eligibility"] == "Manual Review Required"
].copy()

review_columns = [
    "document_id",
    "title",
    "status",
    "source_system",
    "access_level",
    "retrieval_eligibility",
    "review_disposition"
]

display(
    reviewed_records[review_columns].sort_values("document_id")
)

,document_id,title,status,source_system,access_level,retrieval_eligibility,review_disposition
1,DOC-0002,Privileged Access Standard - Working Copy 2,Draft,Controlled Knowledge Repository,Internal,Manual Review Required,Exclude - Draft
2,DOC-0003,Privileged Access Standard - Working Copy 3,Superseded,Controlled Knowledge Repository,Internal,Manual Review Required,Exclude - Superseded
4,DOC-0005,Joiner-Mover-Leaver Procedure - Working Copy 2,Draft,Controlled Knowledge Repository,Internal,Manual Review Required,Exclude - Draft
5,DOC-0006,Joiner-Mover-Leaver Procedure - Working Copy 3,Superseded,Controlled Knowledge Repository,Internal,Manual Review Required,Exclude - Superseded
7,DOC-0008,Service Account Governance - Working Copy 2,Draft,Controlled Knowledge Repository,Internal,Manual Review Required,Exclude - Draft
8,DOC-0009,Service Account Governance - Working Copy 3,Superseded,Controlled Knowledge Repository,Internal,Manual Review Required,Exclude - Superseded
10,DOC-0011,Incident Priority and Escalation Matrix - Working Copy 2,Draft,Controlled Knowledge Repository,Internal,Manual Review Required,Exclude - Draft
11,DOC-0012,Incident Priority and Escalation Matrix - Working Copy 3,Superseded,Controlled Knowledge Repository,Internal,Manual Review Required,Exclude - Superseded
13,DOC-0014,Major Incident Communications SOP - Working Copy 2,Draft,Controlled Knowledge Repository,Internal,Manual Review Required,Exclude - Draft
14,DOC-0015,Major Incident Communications SOP - Working Copy 3,Superseded,Controlled Knowledge Repository,Internal,Manual Review Required,Exclude - Superseded


In [51]:
review_disposition_summary = (
    reviewed_records["review_disposition"]
    .value_counts()
    .rename_axis("review_disposition")
    .reset_index(name="document_count")
)

display(review_disposition_summary)

,review_disposition,document_count
0,Exclude - Draft,24
1,Exclude - Superseded,24
2,Exclude - Irrelevant,2
3,Exclude - Expired,1
4,Exclude - Archived,1
5,Quarantine - Restricted/Conflict,1


In [52]:
review_audit = clean_df[
    clean_df["retrieval_eligibility"] == "Manual Review Required"
].copy()

review_audit.to_csv(
    report_dir / "refined_manual_review_audit.csv",
    index=False
)

print("Refined manual review audit saved.")
print("Records reviewed:", len(review_audit))

Refined manual review audit saved.
Records reviewed: 53


In [53]:
quarantined_document = clean_df[
    clean_df["document_id"] == "DOC-X011"
]

display(
    quarantined_document[
        [
            "document_id",
            "title",
            "status",
            "source_system",
            "access_level",
            "document_text",
            "review_disposition"
        ]
    ]
)

,document_id,title,status,source_system,access_level,document_text,review_disposition
82,DOC-X011,Data Retention - contradictory note,Draft,Personal Drive,Restricted,All business records must be deleted after 30 days. This blanket statement is intentionally inac...,Quarantine - Restricted/Conflict


In [54]:
retrieval_candidates = clean_df[
    clean_df["retrieval_eligibility"] == "Eligible Candidate"
].copy()

print("Total source records:", len(clean_df))
print("Retrieval candidate records:", len(retrieval_candidates))

display(
    retrieval_candidates[
        [
            "document_id",
            "title",
            "category",
            "version",
            "status",
            "source_system",
            "effective_date",
            "review_date",
            "owner",
            "access_level",
            "duplicate_group_id"
        ]
    ].sort_values("document_id")
)

Total source records: 84
Retrieval candidate records: 24


,document_id,title,category,version,status,source_system,effective_date,review_date,owner,access_level,duplicate_group_id
0,DOC-0001,Privileged Access Standard,Access Management,1.1,Approved,Controlled Knowledge Repository,2024-01-24,2025-01-23,Security Owner,Internal,DUP-001
3,DOC-0004,Joiner-Mover-Leaver Procedure,Access Management,1.1,Approved,Controlled Knowledge Repository,2024-02-20,2025-02-19,ITOperations Owner,Internal,DUP-002
6,DOC-0007,Service Account Governance,Access Management,1.1,Approved,Controlled Knowledge Repository,2024-03-18,2025-03-18,Security Owner,Internal,DUP-003
9,DOC-0010,Incident Priority and Escalation Matrix,Incident Management,1.1,Approved,Controlled Knowledge Repository,2024-04-14,2025-04-14,ITServiceDesk Owner,Internal,DUP-004
12,DOC-0013,Major Incident Communications SOP,Incident Management,1.1,Approved,Controlled Knowledge Repository,2024-05-11,2025-05-11,ITOperations Owner,Internal,DUP-005
15,DOC-0016,Endpoint Malware Response Runbook,Incident Management,1.1,Approved,Controlled Knowledge Repository,2024-06-07,2025-06-07,Security Owner,Internal,DUP-006
18,DOC-0019,IT Service Desk SLA,Service Levels,1.1,Approved,Controlled Knowledge Repository,2024-07-04,2025-07-04,ITServiceDesk Owner,Internal,DUP-007
21,DOC-0022,Critical Application Support Agreement,SLA,1.1,Approved,Controlled Knowledge Repository,2024-07-31,2025-07-31,ApplicationSupport Owner,Internal,DUP-008
24,DOC-0025,Vendor Support Escalation Guide,Service Levels,1.1,Approved,Controlled Knowledge Repository,2024-08-27,2025-08-27,VendorManagement Owner,Internal,DUP-009
27,DOC-0028,Production Change Management Policy,Change Management,1.1,Approved,Controlled Knowledge Repository,2024-09-23,2025-09-23,ITOperations Owner,Internal,DUP-010


In [55]:
candidate_missing_summary = (
    retrieval_candidates.isna()
    .sum()
    .reset_index()
)

candidate_missing_summary.columns = [
    "column",
    "missing_count"
]

candidate_missing_summary["missing_percentage"] = (
    candidate_missing_summary["missing_count"]
    / len(retrieval_candidates)
    * 100
)

display(
    candidate_missing_summary[
        candidate_missing_summary["missing_count"] > 0
    ].sort_values("missing_count", ascending=False)
)

,column,missing_count,missing_percentage
4,owner,1,4.166667
6,review_date,1,4.166667


In [56]:
candidate_version_summary = (
    retrieval_candidates[
        ["document_id", "title", "version", "status", "duplicate_group_id"]
    ]
    .sort_values(["duplicate_group_id", "version"])
)

display(candidate_version_summary)

,document_id,title,version,status,duplicate_group_id
0,DOC-0001,Privileged Access Standard,1.1,Approved,DUP-001
3,DOC-0004,Joiner-Mover-Leaver Procedure,1.1,Approved,DUP-002
6,DOC-0007,Service Account Governance,1.1,Approved,DUP-003
9,DOC-0010,Incident Priority and Escalation Matrix,1.1,Approved,DUP-004
12,DOC-0013,Major Incident Communications SOP,1.1,Approved,DUP-005
15,DOC-0016,Endpoint Malware Response Runbook,1.1,Approved,DUP-006
18,DOC-0019,IT Service Desk SLA,1.1,Approved,DUP-007
21,DOC-0022,Critical Application Support Agreement,1.1,Approved,DUP-008
24,DOC-0025,Vendor Support Escalation Guide,1.1,Approved,DUP-009
27,DOC-0028,Production Change Management Policy,1.1,Approved,DUP-010


In [58]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

retrieval_candidates.to_csv(
    processed_dir / "retrieval_candidates.csv",
    index=False
)

print("Retrieval candidate dataset saved successfully.")
print("Saved records:", len(retrieval_candidates))
print("Saved location:", processed_dir)

Retrieval candidate dataset saved successfully.
Saved records: 24
Saved location: C:\Users\main\Desktop\Projects\RAG Based AI Assistant\data\processed


In [59]:
retrieval_candidates["effective_date"] = pd.to_datetime(
    retrieval_candidates["effective_date"],
    errors="coerce"
)

retrieval_candidates["review_date"] = pd.to_datetime(
    retrieval_candidates["review_date"],
    errors="coerce"
)

In [60]:
assessment_date = pd.Timestamp("2026-11-28")

def classify_review_status(row):
    review_date = row["review_date"]

    if pd.isna(review_date):
        return "Missing Review Date"

    if review_date < assessment_date:
        return "Review Overdue"

    return "Review Current"

retrieval_candidates["review_status"] = retrieval_candidates.apply(
    classify_review_status,
    axis=1
)

In [61]:
review_status_summary = (
    retrieval_candidates["review_status"]
    .value_counts()
    .rename_axis("review_status")
    .reset_index(name="document_count")
)

display(review_status_summary)

,review_status,document_count
0,Review Overdue,23
1,Missing Review Date,1


In [62]:
documents_requiring_review = retrieval_candidates[
    retrieval_candidates["review_status"] != "Review Current"
].copy()

display(
    documents_requiring_review[
        [
            "document_id",
            "title",
            "effective_date",
            "review_date",
            "review_status",
            "owner",
            "status"
        ]
    ].sort_values("document_id")
)

,document_id,title,effective_date,review_date,review_status,owner,status
0,DOC-0001,Privileged Access Standard,2024-01-24,2025-01-23,Review Overdue,Security Owner,Approved
3,DOC-0004,Joiner-Mover-Leaver Procedure,2024-02-20,2025-02-19,Review Overdue,ITOperations Owner,Approved
6,DOC-0007,Service Account Governance,2024-03-18,2025-03-18,Review Overdue,Security Owner,Approved
9,DOC-0010,Incident Priority and Escalation Matrix,2024-04-14,2025-04-14,Review Overdue,ITServiceDesk Owner,Approved
12,DOC-0013,Major Incident Communications SOP,2024-05-11,2025-05-11,Review Overdue,ITOperations Owner,Approved
15,DOC-0016,Endpoint Malware Response Runbook,2024-06-07,2025-06-07,Review Overdue,Security Owner,Approved
18,DOC-0019,IT Service Desk SLA,2024-07-04,2025-07-04,Review Overdue,ITServiceDesk Owner,Approved
21,DOC-0022,Critical Application Support Agreement,2024-07-31,2025-07-31,Review Overdue,ApplicationSupport Owner,Approved
24,DOC-0025,Vendor Support Escalation Guide,2024-08-27,2025-08-27,Review Overdue,VendorManagement Owner,Approved
27,DOC-0028,Production Change Management Policy,2024-09-23,2025-09-23,Review Overdue,ITOperations Owner,Approved


In [63]:
documents_requiring_review.to_csv(
    report_dir / "document_freshness_review.csv",
    index=False
)

print("Freshness review report saved.")
print("Documents requiring attention:", len(documents_requiring_review))

Freshness review report saved.
Documents requiring attention: 24


In [64]:
display(
    retrieval_candidates[
        [
            "document_id",
            "title",
            "authority_class",
            "retrieval_eligibility",
            "review_status",
            "owner",
            "review_date"
        ]
    ].head()
)

,document_id,title,authority_class,retrieval_eligibility,review_status,owner,review_date
0,DOC-0001,Privileged Access Standard,Approved Candidate,Eligible Candidate,Review Overdue,Security Owner,2025-01-23
3,DOC-0004,Joiner-Mover-Leaver Procedure,Approved Candidate,Eligible Candidate,Review Overdue,ITOperations Owner,2025-02-19
6,DOC-0007,Service Account Governance,Approved Candidate,Eligible Candidate,Review Overdue,Security Owner,2025-03-18
9,DOC-0010,Incident Priority and Escalation Matrix,Approved Candidate,Eligible Candidate,Review Overdue,ITServiceDesk Owner,2025-04-14
12,DOC-0013,Major Incident Communications SOP,Approved Candidate,Eligible Candidate,Review Overdue,ITOperations Owner,2025-05-11


In [65]:
def assign_governance_status(row):
    if row["review_status"] in ["Review Overdue", "Missing Review Date"]:
        return "Governance Review Required"
    if pd.isna(row["owner"]) or str(row["owner"]).strip() == "":
        return "Metadata Review Required"
    return "Ready for Further Validation"

retrieval_candidates["governance_status"] = retrieval_candidates.apply(
    assign_governance_status,
    axis=1
)

display(
    retrieval_candidates[
        [
            "document_id",
            "title",
            "owner",
            "review_status",
            "governance_status"
        ]
    ]
)

,document_id,title,owner,review_status,governance_status
0,DOC-0001,Privileged Access Standard,Security Owner,Review Overdue,Governance Review Required
3,DOC-0004,Joiner-Mover-Leaver Procedure,ITOperations Owner,Review Overdue,Governance Review Required
6,DOC-0007,Service Account Governance,Security Owner,Review Overdue,Governance Review Required
9,DOC-0010,Incident Priority and Escalation Matrix,ITServiceDesk Owner,Review Overdue,Governance Review Required
12,DOC-0013,Major Incident Communications SOP,ITOperations Owner,Review Overdue,Governance Review Required
15,DOC-0016,Endpoint Malware Response Runbook,Security Owner,Review Overdue,Governance Review Required
18,DOC-0019,IT Service Desk SLA,ITServiceDesk Owner,Review Overdue,Governance Review Required
21,DOC-0022,Critical Application Support Agreement,ApplicationSupport Owner,Review Overdue,Governance Review Required
24,DOC-0025,Vendor Support Escalation Guide,VendorManagement Owner,Review Overdue,Governance Review Required
27,DOC-0028,Production Change Management Policy,ITOperations Owner,Review Overdue,Governance Review Required


In [66]:
governance_summary = (
    retrieval_candidates["governance_status"]
    .value_counts()
    .rename_axis("governance_status")
    .reset_index(name="document_count")
)

display(governance_summary)

,governance_status,document_count
0,Governance Review Required,24


In [67]:
retrieval_candidates.to_csv(
    processed_dir / "retrieval_candidates_with_governance.csv",
    index=False
)

governance_summary.to_csv(
    report_dir / "governance_status_summary.csv",
    index=False
)

print("Governance reports saved successfully.")

Governance reports saved successfully.


In [68]:
content_quality_df = retrieval_candidates[
    [
        "document_id",
        "title",
        "document_text",
        "content_hash",
        "duplicate_group_id",
        "duplicate_record_count",
        "governance_status"
    ]
].copy()

content_quality_df["character_count"] = (
    content_quality_df["document_text"]
    .fillna("")
    .str.len()
)

content_quality_df["word_count"] = (
    content_quality_df["document_text"]
    .fillna("")
    .str.split()
    .str.len()
)

display(
    content_quality_df[
        [
            "document_id",
            "title",
            "character_count",
            "word_count",
            "duplicate_group_id",
            "duplicate_record_count"
        ]
    ]
)

,document_id,title,character_count,word_count,duplicate_group_id,duplicate_record_count
0,DOC-0001,Privileged Access Standard,1025,142,DUP-001,3
3,DOC-0004,Joiner-Mover-Leaver Procedure,1071,146,DUP-002,3
6,DOC-0007,Service Account Governance,999,137,DUP-003,3
9,DOC-0010,Incident Priority and Escalation Matrix,1096,147,DUP-004,3
12,DOC-0013,Major Incident Communications SOP,1083,145,DUP-005,3
15,DOC-0016,Endpoint Malware Response Runbook,1066,146,DUP-006,3
18,DOC-0019,IT Service Desk SLA,1063,152,DUP-007,3
21,DOC-0022,Critical Application Support Agreement,1061,144,DUP-008,3
24,DOC-0025,Vendor Support Escalation Guide,1021,141,DUP-009,3
27,DOC-0028,Production Change Management Policy,1090,148,DUP-010,3


In [69]:
content_length_summary = content_quality_df[
    ["character_count", "word_count"]
].describe().round(2)

display(content_length_summary)

,character_count,word_count
count,24.00,24.00
mean,1047.58,142.58
std,31.81,4.01
min,990.00,137.00
25%,1022.50,139.00
50%,1047.50,142.00
75%,1071.00,145.25
max,1103.00,152.00


In [70]:
content_quality_df["content_issue"] = np.select(
    [
        content_quality_df["word_count"].eq(0),
        content_quality_df["word_count"].lt(50)
    ],
    [
        "Empty Content",
        "Very Short Content"
    ],
    default="No Length Issue Detected"
)

content_issue_summary = (
    content_quality_df["content_issue"]
    .value_counts()
    .rename_axis("content_issue")
    .reset_index(name="document_count")
)

display(content_issue_summary)

,content_issue,document_count
0,No Length Issue Detected,24


In [71]:
content_quality_df.to_csv(
    report_dir / "content_quality_inspection.csv",
    index=False
)

content_issue_summary.to_csv(
    report_dir / "content_issue_summary.csv",
    index=False
)

print("Content quality inspection reports saved.")

Content quality inspection reports saved.
